# Tutorial 1: 16S rRNA Gene Tree Annotation with PyiTOL

**Auto-generate iTOL template files for a 16S rRNA phylogenetic tree with PyiTOL**

**16S rRNA 基因树注释教程——使用 PyiTOL 自动生成 iTOL 模板文件**

This notebook demonstrates how to use PyiTOL's template generation functions to create
iTOL-compatible dataset files for annotating a 16S rRNA phylogenetic tree. We will:

1. Generate a synthetic phylogenetic tree with 50 leaves using DendroPy
2. Create synthetic taxonomy metadata (Phylum, Class, Order, Family, Genus)
3. Generate color-strip, branch-coloring, heatmap, and bar chart iTOL templates
4. Show how to upload the tree and templates to iTOL (simulated)

本教程展示如何使用 PyiTOL 的模板生成功能为 16S rRNA 系统发育树创建 iTOL 数据集文件。我们将：

1. 使用 DendroPy 生成一棵包含 50 个末端（叶）的合成系统发育树
2. 创建合成的分类学元数据（Phylum、Class、Order、Family、Genus）
3. 生出色带、分支着色、热力图与柱状图等 iTOL 模板
4. 演示如何将树和模板上传到 iTOL（模拟）

---

**Prerequisites / 前置条件:**
- `pyitol` (install via `pip install -e .` from the project root) —— 在项目根目录运行 `pip install -e .` 安装
- `dendropy` (included in pyitol dependencies) —— 已包含在 pyitol 的依赖中
- `pandas` (included in pyitol dependencies) —— 已包含在 pyitol 的依赖中

## 1. Setup and Imports

Import the required libraries.

导入所需的库。

In [ ]:
import os
import random
import tempfile
from pathlib import Path

import dendropy
import pandas as pd

# PyiTOL template generators
from pyitol.templates.generator import (
    generate_color_strip_template,
    generate_heatmap_template,
    generate_simple_bar_template,
    generate_branch_template,
)
from pyitol.templates.presets.colorblind import get_colorblind_palette

## 2. Generate a Synthetic Phylogenetic Tree

We build a random 50-leaf ultrametric tree (all root-to-tip distances equal, the shape a
pure-birth/Yule process produces) from an explicit Newick string rather than by calling a
simulator: DendroPy 5 offers no `Tree.purebirth_taxa_tree()` constructor on the `Tree`
class, and its pure-birth simulator lives in `dendropy.simulate.treesim`
(`uniform_pure_birth_tree()`). So the topology is created by recursively splitting the
taxa over 6 tree levels, and branch lengths are then derived from node heights so that
every leaf sits at the same distance from the root. The resulting Newick string is loaded
with `dendropy.Tree.get(data=..., schema="newick")`. The leaf names follow the pattern
`Seq_001` through `Seq_050`.

我们自行构建一棵包含 50 个末端（叶）的随机超度量树（所有根到末端距离相等，即纯出生/Yule
过程的典型形状）：这里不调用随机树生成器，而是显式写出 Newick 字符串——DendroPy 5 的
`Tree` 类上并没有 `Tree.purebirth_taxa_tree()` 这样的构造函数，其纯出生（pure-birth）
模拟器位于 `dendropy.simulate.treesim`（`uniform_pure_birth_tree()`）。因此先用 6 层递归
拆分末端名称得到拓扑结构，再依据内部节点高度计算分支长度，使每个末端（叶）到根的距离保持
一致；最后用 `dendropy.Tree.get(data=..., schema="newick")` 读入该 Newick 字符串。
末端（叶）名称遵循 `Seq_001` 至 `Seq_050` 的规则。

In [ ]:
# Set random seed for reproducibility
random.seed(42)

# DendroPy 5 offers no pure-birth/Yule tree generator on the `Tree` class (that
# simulator lives in dendropy.simulate.treesim.uniform_pure_birth_tree), so we
# build the 50-tip ultrametric tree ourselves:
#   1. recursively split the taxa into two groups over N_LEVELS tree levels;
#   2. place every internal node one level above its tallest child, so each
#      branch length is (parent height - child height) and all root-to-tip
#      distances are equal (ultrametric by construction).
N_LEVELS = 6  # 2**6 = 64 >= 50 tips
UNIT = 0.1


def build_clade(labels, levels):
    """Return the Newick substring for `labels`, split at most `levels` deep."""
    if len(labels) == 1:
        return labels[0]
    capacity = 2 ** (levels - 1)
    k = random.randint(max(1, len(labels) - capacity), min(len(labels) - 1, capacity))
    return f"({build_clade(labels[:k], levels - 1)},{build_clade(labels[k:], levels - 1)})"


leaf_labels = [f"Seq_{i:03d}" for i in range(1, 51)]
newick = build_clade(leaf_labels, N_LEVELS) + ";"
tree = dendropy.Tree.get(data=newick, schema="newick", preserve_underscores=True)


def node_height(node):
    """Leaves sit at height 0; an internal node is one level above its tallest child."""
    if node.is_leaf():
        return 0.0
    return max(node_height(child) for child in node.child_nodes()) + 1.0


def assign_ultrametric_lengths(node):
    """Set edge lengths from node heights so the tree is ultrametric."""
    for child in node.child_nodes():
        child.edge_length = (node_height(node) - node_height(child)) * UNIT
        assign_ultrametric_lengths(child)


assign_ultrametric_lengths(tree.seed_node)

# Write to a temporary Newick file
tmpdir = Path(tempfile.mkdtemp(prefix="pyitol_tutorial_"))
tree_path = tmpdir / "16s_rrna_tree.nwk"
tree.write_to_path(str(tree_path), schema="newick")

print(f"Tree written to: {tree_path}")
print(f"Number of leaves: {len(tree.leaf_nodes())}")
print(f"First 10 leaf names: {[leaf.taxon.label for leaf in tree.leaf_node_iter()][:10]}")

## 3. Create Synthetic Taxonomy Metadata

We simulate a realistic 16S rRNA taxonomy table with the following columns:
- `id`: Sequence identifier (matches tree leaf names)
- `Phylum`: Bacterial phylum (e.g., Proteobacteria, Firmicutes)
- `Class`: Taxonomic class
- `Order`: Taxonomic order
- `Family`: Taxonomic family
- `Genus`: Taxonomic genus
- `Abundance`: Simulated relative abundance (numeric, for heatmap/bar)
- `GC_Content`: Simulated GC content percentage (numeric)

我们模拟一张符合真实情况的 16S rRNA 分类学元数据表，包含以下各列：
- `id`：序列标识（与树的末端（叶）名称一致）
- `Phylum`：细菌门（例如 Proteobacteria、Firmicutes）
- `Class`：分类学纲
- `Order`：分类学目
- `Family`：分类学科
- `Genus`：分类学属
- `Abundance`：模拟的相对丰度（数值型，用于热力图和柱状图）
- `GC_Content`：模拟的 GC 含量百分比（数值型）

In [ ]:
# Define a realistic taxonomy hierarchy
taxonomy_hierarchy = {
    "Proteobacteria": {
        "Gammaproteobacteria": {
            "Enterobacterales": {
                "Enterobacteriaceae": ["Escherichia", "Salmonella", "Klebsiella"],
                "Yersiniaceae": ["Serratia"],
            },
            "Pseudomonadales": {
                "Pseudomonadaceae": ["Pseudomonas"],
                "Moraxellaceae": ["Acinetobacter"],
            },
        },
        "Betaproteobacteria": {
            "Burkholderiales": {
                "Burkholderiaceae": ["Burkholderia", "Ralstonia"],
            },
        },
    },
    "Firmicutes": {
        "Bacilli": {
            "Bacillales": {
                "Bacillaceae": ["Bacillus"],
                "Staphylococcaceae": ["Staphylococcus"],
            },
            "Lactobacillales": {
                "Lactobacillaceae": ["Lactobacillus"],
                "Streptococcaceae": ["Streptococcus"],
            },
        },
    },
    "Actinobacteria": {
        "Actinomycetia": {
            "Corynebacteriales": {
                "Corynebacteriaceae": ["Corynebacterium"],
                "Mycobacteriaceae": ["Mycobacterium"],
            },
        },
    },
}

# Flatten the hierarchy into a list of (Phylum, Class, Order, Family, Genus) tuples
taxonomy_records = []
for phylum, classes in taxonomy_hierarchy.items():
    for cls, orders in classes.items():
        for order, families in orders.items():
            for family, genera in families.items():
                for genus in genera:
                    taxonomy_records.append((phylum, cls, order, family, genus))

# Assign taxonomy to each leaf, distributing roughly evenly
leaf_names = [leaf.taxon.label for leaf in tree.leaf_node_iter()]
records = []
for i, name in enumerate(leaf_names):
    rec = taxonomy_records[i % len(taxonomy_records)]
    records.append({
        "id": name,
        "Phylum": rec[0],
        "Class": rec[1],
        "Order": rec[2],
        "Family": rec[3],
        "Genus": rec[4],
        "Abundance": round(random.uniform(0.1, 15.0), 2),
        "GC_Content": round(random.uniform(35.0, 65.0), 1),
    })

taxonomy_df = pd.DataFrame(records)

# Save to CSV
taxonomy_path = tmpdir / "taxonomy.csv"
taxonomy_df.to_csv(taxonomy_path, index=False)

print(f"Taxonomy saved to: {taxonomy_path}")
print(f"Shape: {taxonomy_df.shape}")
taxonomy_df.head(10)

## 4. Generate Color-Strip Template

The color-strip template maps each Phylum to a color and displays it as a colored strip
next to the tree. This is one of the most commonly used iTOL annotation types. The cells
below first build the color strip for Phylum, and then build a branch-coloring template
for the same column.

色带模板将每个 Phylum（门）映射为一种颜色，并以彩色条带的形式显示在树的旁边。
这是最常用的 iTOL 注释类型之一。下面的单元格先为 Phylum 生出色带模板，
随后再为同一列生成分支着色模板。

In [ ]:
# Get a colorblind-friendly palette for phyla
phyla = sorted(taxonomy_df["Phylum"].unique())
palette = get_colorblind_palette("tol_bright", n=len(phyla))
phylum_colors = dict(zip(phyla, palette))

print("Phylum color mapping:")
for phylum, color in phylum_colors.items():
    print(f"  {phylum}: {color}")

In [ ]:
# Generate color-strip template for Phylum
color_strip_path = tmpdir / "phylum_color_strip.txt"

generate_color_strip_template(
    output_path=color_strip_path,
    taxonomy_path=taxonomy_path,
    tree_path=tree_path,
    column="Phylum",
    colors=phylum_colors,
    label="Phylum",
    strip_width="60",
)

print(f"Color-strip template written to: {color_strip_path}")
print("--- Preview (first 30 lines) ---")
print(color_strip_path.read_text()[:800])

In [ ]:
# Also generate a branch coloring template by Phylum
branch_path = tmpdir / "phylum_branch_colors.txt"

generate_branch_template(
    output_path=branch_path,
    taxonomy_path=taxonomy_path,
    tree_path=tree_path,
    column="Phylum",
    colors=phylum_colors,
    label="Phylum_branches",
)

print(f"Branch-coloring template written to: {branch_path}")
print("--- Preview (first 20 lines) ---")
print(branch_path.read_text()[:600])

## 5. Generate Heatmap Template

A heatmap template visualizes numeric data (Abundance, GC_Content) as a color gradient
alongside the tree. We use a red-yellow-blue gradient (`#d73027`, `#ffffbf`, `#4575b4`).

热力图模板将数值数据（Abundance、GC_Content）以颜色渐变的形式显示在树的旁边。
这里使用红-黄-蓝三色渐变（`#d73027`、`#ffffbf`、`#4575b4`）。

In [ ]:
heatmap_path = tmpdir / "abundance_heatmap.txt"

generate_heatmap_template(
    output_path=heatmap_path,
    taxonomy_path=taxonomy_path,
    tree_path=tree_path,
    value_columns=["Abundance", "GC_Content"],
    color_gradient=["#d73027", "#ffffbf", "#4575b4"],
    label="Abundance_GC",  
    column_labels={"Abundance": "Rel. Abundance", "GC_Content": "GC %"},
)

print(f"Heatmap template written to: {heatmap_path}")
print("--- Preview (first 30 lines) ---")
print(heatmap_path.read_text()[:800])

## 6. Generate Bar Chart Template

A simple bar chart template displays a single numeric value (Abundance) as horizontal bars
next to each leaf.

简单柱状图模板将单一数值（Abundance）以水平长条的形式显示在每个末端（叶）的旁边。

In [ ]:
bar_path = tmpdir / "abundance_bar.txt"

generate_simple_bar_template(
    output_path=bar_path,
    taxonomy_path=taxonomy_path,
    tree_path=tree_path,
    value_column="Abundance",
    label="Abundance_bar",
    bar_color="#3c5484",
    bar_width="80",
)

print(f"Bar chart template written to: {bar_path}")
print("--- Preview (first 25 lines) ---")
print(bar_path.read_text()[:600])

## 7. Review Generated Files

Let's list all the template files we've generated.

列出我们生成的所有模板文件。

In [ ]:
generated_files = list(tmpdir.glob("*.txt")) + list(tmpdir.glob("*.nwk")) + list(tmpdir.glob("*.csv"))
print("Generated files:")
for f in sorted(generated_files):
    size = f.stat().st_size
    print(f"  {f.name:30s}  {size:>6d} bytes")

print(f"\nAll files are in: {tmpdir}")

## 8. Upload to iTOL (Simulated)

In a real workflow, you would use `ITOLAPIClient` to upload the tree and templates to iTOL.
Below we show the exact code, but the actual API call is commented out since it requires
a valid API key.

在实际工作流中，您可以使用 `ITOLAPIClient` 将树和模板上传到 iTOL。
下面给出了完整的代码，但由于需要有效的 API Key，实际的 API 调用已被注释掉。

In [ ]:
from pyitol.api.client import ITOLAPIClient

# --- Simulated upload workflow ---
# In a real scenario, you would do:
#
# client = ITOLAPIClient(api_key_file=".itolapi.key")
# tree_id = client.upload(
#     tree_file=str(tree_path),
#     template_files=[
#         str(color_strip_path),
#         str(branch_path),
#         str(heatmap_path),
#         str(bar_path),
#     ],
#     force=True,
# )
# print(f"Tree uploaded. iTOL tree ID: {tree_id}")
# print(f"View at: https://itol.embl.de/tree/{tree_id}")
#
# # Export as PDF
# result = client.export([tree_id], fmt="pdf", output_dir=str(tmpdir))
# print(f"Exported: {result}")

# For this tutorial, we simulate the response:
print("[Simulated] Upload would send:")
print(f"  Tree file:    {tree_path.name}")
print(f"  Templates:    {[f.name for f in [color_strip_path, branch_path, heatmap_path, bar_path]]}")
print(f"  Tree ID:      simulated_tree_id_12345")
print(f"  View URL:     https://itol.embl.de/tree/simulated_tree_id_12345")

## 9. Summary

In this tutorial we demonstrated the core PyiTOL workflow for 16S rRNA tree annotation:

1. **Tree generation**: Built a synthetic 50-leaf ultrametric tree by writing an explicit Newick string and assigning branch lengths from node heights (DendroPy 5 exposes no pure-birth/Yule tree generator on the `Tree` class; that simulator lives in `dendropy.simulate.treesim`)
2. **Taxonomy metadata**: Created a CSV with hierarchical taxonomy and numeric data
3. **Color-strip template**: `generate_color_strip_template()` for Phylum-level coloring
4. **Branch coloring**: `generate_branch_template()` for clade-based branch colors
5. **Heatmap template**: `generate_heatmap_template()` for multi-column numeric data
6. **Bar chart template**: `generate_simple_bar_template()` for single-value bars
7. **iTOL upload**: Shown the upload workflow (simulated)

本教程演示了 PyiTOL 用于 16S rRNA 树注释的核心工作流：

1. **树的生成**：通过显式写出 Newick 字符串并按节点高度赋予分支长度，构建了一棵包含 50 个末端（叶）的合成超度量树（DendroPy 5 未在 `Tree` 类上提供纯出生/Yule 树生成器，该模拟器位于 `dendropy.simulate.treesim`）
2. **分类学元数据**：创建了包含层级分类信息与数值数据的 CSV 文件
3. **色带模板**：使用 `generate_color_strip_template()` 按 Phylum 着色
4. **分支着色**：使用 `generate_branch_template()` 按单系群为分支着色
5. **热力图模板**：使用 `generate_heatmap_template()` 展示多列数值数据
6. **柱状图模板**：使用 `generate_simple_bar_template()` 展示单值柱形
7. **iTOL 上传**：演示了上传工作流（模拟）

---

### Next Steps / 后续步骤

- See `02_virus_monophyly.ipynb` for monophyly analysis —— 单系性分析请见 `02_virus_monophyly.ipynb`
- See `03_batch_workflow.ipynb` for batch processing multiple trees —— 多棵树的批处理请见 `03_batch_workflow.ipynb`

In [ ]:
# Cleanup temporary directory (optional)
# Uncomment the next line to remove generated files:
# import shutil; shutil.rmtree(tmpdir)
print(f"Tutorial complete. Temporary files are in: {tmpdir}")